# FoodTech Capstone — Phase 8: Hypothesis Testing & Statistical Validation
### Data Science Capstone | Topic 7 (NIIT) | *Analyze Food Delivery Demand Across Regions to Inform Market Expansion & Logistics*

This notebook  works through the four stages set out for this phase —
**Hypothesis Formulation → Statistical Test Execution → Output Interpretation → Validation Summary** —
for every one of the **20 business questions / hypotheses** defined in the Phase 3 document
(`Phase3_Objectives_Questions_Hypotheses.docx`).


| Objective | Focus | Questions |
|---|---|---|
| 1 | Regional Demand & Market Expansion Potential | Q1–Q4 |
| 2 | Customer Behavior, Segmentation & Loyalty | Q5–Q10 |
| 3 | Delivery Partner & Logistics Efficiency | Q11–Q14 |
| 4 | Restaurant Performance & Operational Capacity | Q15–Q18 |
| 5 | Order Economics, Discounts & Profitability Risk | Q19–Q20 |

**Significance threshold used throughout:** α = 0.05.


##  Setup — Imports & Data Loading
Place the four cleaned data files (`Orders_Cleaned.xlsx`, `Customers_Cleaned.xlsx`, `Delivery_Partners_Cleaned.xlsx`, `Restaurant_Cleaned.xlsx`) in the same folder as this notebook before running.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import pearsonr, ttest_ind, chi2_contingency, f_oneway
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
ALPHA = 0.05
print('Environment ready. Significance level (alpha) =', ALPHA)

Environment ready. Significance level (alpha) = 0.05


In [2]:
orders      = pd.read_excel('Orders_Cleaned.xlsx', sheet_name='Orders_Data')
customers   = pd.read_excel('Customers_Cleaned.xlsx')
partners    = pd.read_excel('Delivery_Partners_Cleaned.xlsx')
restaurants = pd.read_excel('Restaurant_Cleaned.xlsx')

for name, df in [('Orders', orders), ('Customers', customers),
                  ('Delivery Partners', partners), ('Restaurants', restaurants)]:
    print(f'{name:20s} -> {df.shape[0]:>7,} rows x {df.shape[1]} columns')

Orders               -> 100,000 rows x 23 columns
Customers            ->  22,000 rows x 10 columns
Delivery Partners    ->   1,100 rows x 10 columns
Restaurants          ->     550 rows x 8 columns


In [3]:
orders['Region'] = orders['Region'].replace({'Metropolitian': 'Metropolitan'})
orders['Order_Date'] = pd.to_datetime(orders['Order_Date'])
orders['Month_Year'] = orders['Order_Date'].dt.to_period('M').astype(str)

customers_r   = customers.rename(columns={'Customer_Id': 'Customer_ID', 'City': 'Region'})
restaurants_r = restaurants.rename(columns={'City': 'Rest_Region'})

master = (orders
          .merge(customers_r, on='Customer_ID', how='left', suffixes=('', '_cust'))
          .merge(restaurants_r, on='Restaurant_ID', how='left', suffixes=('', '_rest'))
          .merge(partners, on='Delivery_Partner_ID', how='left', suffixes=('', '_partner')))

print('Master analytical table:', master.shape)

Master analytical table: (100000, 49)


---
# Objective 1 — Regional Demand & Market Expansion Potential
Identify which regions generate the highest order volume, revenue, and growth momentum, to prioritize where FoodTech should expand next.

## Q1 & Q2 — Region vs Order Value / Revenue
**Business question:** Which regions generate the highest order volume/revenue, and how does average order value differ across regions?

**Hypothesis Formulation**
- Test: One-way ANOVA (two-tailed)
- H0: Mean `Final_Amount` is equal across Urban, Semi-Urban and Metropolitan regions.
- H1: Mean `Final_Amount` differs across at least one region.

In [4]:
order_volume  = master['Region'].value_counts()
region_revenue = master.groupby('Region')['Net_Revenue'].sum().sort_values(ascending=False)
print('Order volume by region:'); print(order_volume)
print('\nNet revenue by region:'); print(region_revenue.round(0))

groups = [g['Final_Amount'].dropna().values for _, g in master.groupby('Region')]
f_stat, p_val = f_oneway(*groups)
print(f'\nOne-way ANOVA -> F = {f_stat:.3f}, p = {p_val:.4g}')

Order volume by region:
Region
Urban           33421
Semi-Urban      33349
Metropolitan    33230
Name: count, dtype: int64

Net revenue by region:
Region
Urban           17434107
Semi-Urban      17355192
Metropolitan    17273586
Name: Net_Revenue, dtype: int64



One-way ANOVA -> F = 1.053, p = 0.3488


**Output Interpretation:** F = 1.053, p = 0.3488. Since p (0.3488) > α (0.05), the differences in
average `Final_Amount` between regions are not statistically significant — the ANOVA F-statistic is close
to 1, indicating the between-region variance is no larger than expected by chance.

**Validation Summary**
**Decision: Fail to reject H0 (retain H0)** — average order value does **not** differ meaningfully by region. However, raw
order volume and total net revenue still lead in one region (see counts above), so expansion priority
should be driven by *volume/revenue scale*, not by any per-order value premium in a particular region.

## Q3 — Region vs Repeat-Order Growth Momentum
**Hypothesis Formulation**
- Test: Linear trend regression (slope), one-tailed (right), per region
- H0: There is no upward month-over-month growth trend in order volume within a region (slope = 0).
- H1: The region shows a significant positive growth trend over time (slope > 0).

In [5]:
region_month = master.groupby(['Month_Year', 'Region']).size().unstack('Region').sort_index()

trend_rows = []
for region in region_month.columns:
    y = region_month[region].values
    x = np.arange(len(y))
    slope, intercept, r, p_two, se = stats.linregress(x, y)
    p_one = p_two / 2 if slope > 0 else 1 - p_two / 2
    trend_rows.append((region, slope, p_one))
    print(f'{region:15s} slope={slope:+.3f} orders/month, one-tailed p={p_one:.4g}')

Metropolitan    slope=+1.698 orders/month, one-tailed p=0.07433
Semi-Urban      slope=+1.766 orders/month, one-tailed p=0.0669
Urban           slope=+1.690 orders/month, one-tailed p=0.07934


**Output Interpretation:** All three regions show a small positive slope (~+1.7 orders/month) but none
clears the one-tailed significance threshold (one-tailed p ≈ 0.067–0.079, all > 0.05).

**Validation Summary**
**Decision: Fail to reject H0 for all three regions.** There is no statistically significant month-over-month
growth trend in any region over the observed window — order volume is essentially flat across regions, so
region selection for expansion should rely on current scale (Q1) and festival responsiveness (Q4) rather
than an already-emerging growth trend.

## Q4 — Festival_Flag vs Order Volume, by Region
**Hypothesis Formulation**
- Test: Independent-samples t-test (one-tailed, right), run separately for each region
- H0: Mean daily order volume is equal on festival vs non-festival days.
- H1: Mean daily order volume is greater on festival days.

In [6]:
for region in master['Region'].dropna().unique():
    sub = master[master['Region'] == region]
    daily_fest = sub[sub['Festival_Flag'] == True].groupby(sub['Order_Date'].dt.date).size()
    daily_reg  = sub[sub['Festival_Flag'] == False].groupby(sub['Order_Date'].dt.date).size()
    t_stat, p_val = ttest_ind(daily_fest, daily_reg, equal_var=False)
    p_one = p_val / 2 if t_stat > 0 else 1 - p_val / 2
    print(f'{region:15s} t={t_stat:+.2f}, one-tailed p={p_one:.4g} ->',
          'significant festival uplift' if p_one < ALPHA else 'no significant uplift')

Metropolitan    t=+9.96, one-tailed p=1e-22 -> significant festival uplift
Semi-Urban      t=+9.90, one-tailed p=1.731e-22 -> significant festival uplift


Urban           t=+10.40, one-tailed p=1.55e-24 -> significant festival uplift


**Output Interpretation:** All three regions show large positive t-statistics (~+9.9 to +10.4) with
p-values effectively 0 (order of 1e-22 to 1e-24) — far below α = 0.05.

**Validation Summary**
**Decision: Reject H0 for all three regions.** Festival days drive a statistically significant increase in
daily order volume everywhere, with a similar magnitude across regions — festival-period logistics/staffing
should be scaled up uniformly rather than concentrated in one region.

---
# Objective 2 — Customer Behavior, Segmentation & Loyalty
Understand how customer type, demographics, and cuisine preference drive spend and retention.

## Q5 — Customer_Type vs Average_Spend
**Hypothesis Formulation**
- Test: One-way ANOVA (one-tailed)
- H0: Mean `Average_Spend` is equal across New, Returning and Loyal customers.
- H1: Mean `Average_Spend` differs, expected Loyal > Returning > New.

In [7]:
spend_by_type = customers.groupby('Customer_Type')['Average_Spend'].mean().sort_values(ascending=False)
print(spend_by_type.round(2))

groups5 = [g['Average_Spend'].dropna().values for _, g in customers.groupby('Customer_Type')]
f5, p5 = f_oneway(*groups5)
print(f'\nOne-way ANOVA -> F = {f5:.3f}, p = {p5:.4g}')

Customer_Type
Loyal       342.8800
New         326.6600
Returning   312.5300
Name: Average_Spend, dtype: float64

One-way ANOVA -> F = 199.798, p = 1.018e-86


**Output Interpretation:** F = 199.80, p = 1.02e-86 — extremely small p-value, far below α = 0.05.
Mean spend ranking: Loyal (₹342.9) > New (₹326.7) > Returning (₹312.5).

**Validation Summary**
**Decision: Reject H0.** Average spend differs significantly by customer type. Loyal customers do spend the
most as hypothesized, but Returning customers spend *less* than New customers — the expected strict
ordering (Loyal > Returning > New) is only partly confirmed; the Returning segment looks like a retention
gap worth targeted marketing.

## Q6 — Preferred_Cuisine vs Customer_Type
**Hypothesis Formulation**
- Test: Chi-square test of independence (two-tailed)
- H0: `Preferred_Cuisine` is independent of `Customer_Type`.
- H1: `Preferred_Cuisine` is associated with `Customer_Type`.

In [8]:
ct6 = pd.crosstab(customers['Customer_Type'], customers['Preferred_Cuisine'])
chi6, p6, dof6, exp6 = chi2_contingency(ct6)
print(ct6)
print(f'\nChi-square = {chi6:.2f}, dof = {dof6}, p = {p6:.4g}')

Preferred_Cuisine  Chinese  Desserts  Fast Food  Italian  North Indian  \
Customer_Type                                                            
Loyal                  734       691        730      768           736   
New                   1158      1006       1139     1086          1127   
Returning             1901      1837       1838     1718          1913   

Preferred_Cuisine  South Indian  
Customer_Type                    
Loyal                       741  
New                        1084  
Returning                  1793  

Chi-square = 16.23, dof = 10, p = 0.09321


**Output Interpretation:** χ² = 16.23, dof = 10, p = 0.0932 — above α = 0.05, so the association is not
strong enough to be statistically significant at the 5% level.

**Validation Summary**
**Decision: Fail to reject H0.** Cuisine preference does not vary significantly by customer type in this
dataset — cuisine-based marketing does not need to be differentiated by loyalty segment.

## Q7 — Total_Orders vs Average_Spend
**Hypothesis Formulation**
- Test: Pearson correlation (one-tailed, right)
- H0: No significant correlation exists (ρ = 0).
- H1: A significant positive correlation exists (ρ > 0).

In [9]:
r7, p7 = pearsonr(customers['Total_Orders'], customers['Average_Spend'])
p7_one = p7 / 2 if r7 > 0 else 1 - p7 / 2
print(f'Pearson r = {r7:.4f}, one-tailed p = {p7_one:.4g}')

Pearson r = 0.0481, one-tailed p = 4.486e-13


**Output Interpretation:** r = 0.048 (very weak positive), one-tailed p = 4.49e-13 — statistically
significant only because of the large sample size (n = 22,000); the effect size itself is negligible.

**Validation Summary**
**Decision: Reject H0**, but with a caveat: the correlation is statistically significant yet practically
very weak (r ≈ 0.05). Order frequency alone is a poor predictor of spend level — spend is driven more by
customer type/segment (Q5) than by order count.

## Q8 — Customer_Type vs Order_Rating
**Hypothesis Formulation**
- Test: One-way ANOVA (two-tailed)
- H0: Mean `Order_Rating` is equal across customer segments.
- H1: Mean `Order_Rating` differs across at least one segment.

In [10]:
groups8 = [g['Order_Rating'].dropna().values for _, g in master.groupby('Customer_Type')]
f8, p8 = f_oneway(*groups8)
print(master.groupby('Customer_Type')['Order_Rating'].mean().round(3))
print(f'\nOne-way ANOVA -> F = {f8:.3f}, p = {p8:.4g}')

Customer_Type
Loyal       4.0330
New         4.0240
Returning   4.0190
Name: Order_Rating, dtype: float64

One-way ANOVA -> F = 1.107, p = 0.3306


**Output Interpretation:** F = 1.107, p = 0.3306 — above α = 0.05.

**Validation Summary**
**Decision: Fail to reject H0.** Order ratings are consistent across New/Returning/Loyal customers —
satisfaction is not a segment-specific issue.

## Q9 — Payment_Mode vs Customer_Type
**Hypothesis Formulation**
- Test: Chi-square test of independence (two-tailed)
- H0: `Payment_Mode` choice is independent of `Customer_Type`.
- H1: `Payment_Mode` choice is associated with `Customer_Type`.

In [11]:
ct9 = pd.crosstab(master['Customer_Type'], master['Payment_Mode'])
chi9, p9, dof9, exp9 = chi2_contingency(ct9)
print(ct9)
print(f'\nChi-square = {chi9:.2f}, dof = {dof9}, p = {p9:.4g}')

Payment_Mode    Card  Cash    UPI  Wallet
Customer_Type                            
Loyal           2722  1095   5548    1695
New            13528  5359  26990    8190
Returning       8713  3522  17440    5198

Chi-square = 2.88, dof = 6, p = 0.8241


**Output Interpretation:** χ² = 2.88, dof = 6, p = 0.8241 — well above α = 0.05.

**Validation Summary**
**Decision: Fail to reject H0.** Payment mode preference is uniform across customer segments — no need to
tailor payment-method promotions by loyalty tier.

## Q10 — Age_Group vs Preferred_Cuisine
**Hypothesis Formulation**
- Test: Chi-square test of independence (two-tailed)
- H0: `Preferred_Cuisine` is independent of `Age_Group`.
- H1: `Preferred_Cuisine` is associated with `Age_Group`.

In [12]:
ct10 = pd.crosstab(customers['Age_Group'], customers['Preferred_Cuisine'])
chi10, p10, dof10, exp10 = chi2_contingency(ct10)
print(ct10)
print(f'\nChi-square = {chi10:.2f}, dof = {dof10}, p = {p10:.4g}')

Preferred_Cuisine  Chinese  Desserts  Fast Food  Italian  North Indian  \
Age_Group                                                                
18-25                  859       765        794      776           797   
26-35                  949       972       1090      956          1067   
36-50                 1546      1450       1460     1469          1440   
50+                    439       347        363      371           472   

Preferred_Cuisine  South Indian  
Age_Group                        
18-25                       817  
26-35                       981  
36-50                      1400  
50+                         420  

Chi-square = 46.71, dof = 15, p = 4.098e-05


**Output Interpretation:** χ² = 46.71, dof = 15, p = 4.10e-05 — well below α = 0.05.

**Validation Summary**
**Decision: Reject H0.** Cuisine preference **is** significantly associated with age group — this is the
one demographic factor (unlike customer type in Q6) worth using for age-targeted cuisine promotions.

---
# Objective 3 — Delivery Partner & Logistics Efficiency
Evaluate delivery partner performance across vehicle types and regions to optimize logistics routing and partner allocation.

## Q11 — Vehicle_Type vs Avg_Delivery_Speed_KMPH
**Hypothesis Formulation**
- Test: One-way ANOVA (two-tailed)
- H0: Mean `Avg_Delivery_Speed_KMPH` is equal across all vehicle types.
- H1: Mean `Avg_Delivery_Speed_KMPH` differs by vehicle type.

In [13]:
groups11 = [g['Avg_Delivery_Speed_KMPH'].dropna().values for _, g in partners.groupby('Vehicle_Type')]
f11, p11 = f_oneway(*groups11)
print(partners.groupby('Vehicle_Type')['Avg_Delivery_Speed_KMPH'].mean().round(2))
print(f'\nOne-way ANOVA -> F = {f11:.3f}, p = {p11:.4g}')

Vehicle_Type
Bicycle   22.5000
Bike      22.0700
Car       22.1300
Scooter   22.7200
Name: Avg_Delivery_Speed_KMPH, dtype: float64

One-way ANOVA -> F = 1.058, p = 0.3662


**Output Interpretation:** F = 1.058, p = 0.3662 — above α = 0.05.

**Validation Summary**
**Decision: Fail to reject H0.** Delivery speed does not differ significantly by vehicle type — routing
decisions should weight cost/availability over an assumed speed advantage of one vehicle type.

## Q12 — Delivery_Efficiency_Score vs Avg_Customer_Rating
**Hypothesis Formulation**
- Test: Pearson correlation (one-tailed, right)
- H0: No significant correlation exists (ρ = 0).
- H1: A significant positive correlation exists (ρ > 0).

In [14]:
r12, p12 = pearsonr(partners['Delivery_Efficiency_Score'], partners['Avg_Customer_Rating'])
p12_one = p12 / 2 if r12 > 0 else 1 - p12 / 2
print(f'Pearson r = {r12:.4f}, one-tailed p = {p12_one:.4g}')

Pearson r = 0.3764, one-tailed p = 1.218e-38


**Output Interpretation:** r = 0.376 (moderate positive), one-tailed p = 1.22e-38 — well below α = 0.05.

**Validation Summary**
**Decision: Reject H0.** Delivery efficiency score is a meaningfully strong, significant predictor of
customer rating — investing in partner efficiency should measurably improve ratings.

## Q13 — Region vs Delayed_Deliveries
**Hypothesis Formulation**
- Test: Chi-square test of independence (two-tailed), on binned delay-rate categories per partner
- H0: Delay rate is independent of `Region`.
- H1: Delay rate is associated with `Region`.

In [15]:
partners_scored = partners.copy()
partners_scored['Delay_Rate'] = partners_scored['Delayed_Deliveries'] / (
    partners_scored['Successful_Deliveries'] + partners_scored['Delayed_Deliveries'])

region_delay = partners_scored.groupby('Region')['Delay_Rate'].mean().sort_values(ascending=False)
print('Average delay rate by region:'); print((region_delay * 100).round(2).astype(str) + '%')

delay_ct = pd.crosstab(partners_scored['Region'],
                        pd.cut(partners_scored['Delay_Rate'], bins=[-0.01, 0.1, 0.2, 1], labels=['Low', 'Medium', 'High']))
chi13, p13, dof13, exp13 = chi2_contingency(delay_ct)
print(f'\nChi-square = {chi13:.2f}, dof = {dof13}, p = {p13:.4g}')

Average delay rate by region:
Region
Semi-Urban      100.0%
Metropolitan    11.41%
Urban            4.97%
Name: Delay_Rate, dtype: str

Chi-square = 608.21, dof = 4, p = 2.594e-130


**Output Interpretation:** χ² = 608.21, dof = 4, p = 2.59e-130 — extremely far below α = 0.05. The
regional averages above show why: Semi-Urban partners have a mean delay rate of 100% (every single
Semi-Urban partner is fully delayed), versus ~11% for Metropolitan and ~5% for Urban — an unusually clean
split that is worth a data-quality check, not just a business one.

**Validation Summary**
**Decision: Reject H0.** Delay rate is strongly associated with region. Given that the Semi-Urban figure is
100% across every partner in that region (not just "higher"), this should first be verified against the
source system before acting on it — if confirmed, Semi-Urban logistics needs urgent intervention; if it
turns out to be a data-capture artifact, the field should be corrected before this result is used to guide
partner allocation.

## Q14 — Delivery_Time_Minutes vs Order_Rating
**Hypothesis Formulation**
- Test: Pearson correlation (one-tailed, left)
- H0: No significant correlation exists (ρ = 0).
- H1: A significant negative correlation exists (ρ < 0) — longer delivery time associates with lower rating.

In [16]:
valid = master.dropna(subset=['Delivery_Time_Minutes', 'Order_Rating'])
r14, p14 = pearsonr(valid['Delivery_Time_Minutes'], valid['Order_Rating'])
p14_one = p14 / 2 if r14 < 0 else 1 - p14 / 2
print(f'Pearson r = {r14:.4f}, one-tailed p = {p14_one:.4g}')

Pearson r = -0.0022, one-tailed p = 0.2442


**Output Interpretation:** r = -0.0022 (essentially zero), one-tailed p = 0.244 — above α = 0.05.

**Validation Summary**
**Decision: Fail to reject H0.** Delivery time shows no meaningful linear relationship with order rating in
this dataset — rating is likely driven by other factors (food quality, order accuracy) rather than raw
delivery speed.

---
# Objective 4 — Restaurant Performance & Operational Capacity
Assess restaurant-level preparation time, ratings, and order capacity utilization to identify high-performing and under-capacity restaurants.

## Q15 — Avg_Preparation_Time_Minutes vs Avg_Rating
**Hypothesis Formulation**
- Test: Pearson correlation (one-tailed, left)
- H0: No significant correlation exists (ρ = 0).
- H1: A significant negative correlation exists (ρ < 0).

In [17]:
r15, p15 = pearsonr(restaurants['Avg_Preparation_Time_Minutes'], restaurants['Avg_Rating'])
p15_one = p15 / 2 if r15 < 0 else 1 - p15 / 2
print(f'Pearson r = {r15:.4f}, one-tailed p = {p15_one:.4g}')

Pearson r = -0.3058, one-tailed p = 1.137e-13


**Output Interpretation:** r = -0.306 (moderate negative), one-tailed p = 1.14e-13 — well below α = 0.05.

**Validation Summary**
**Decision: Reject H0.** Longer preparation time is significantly associated with lower restaurant ratings —
kitchen/prep-time optimization is a legitimate lever for improving ratings.

## Q16 — Cuisine_Type vs Avg_Rating
**Hypothesis Formulation**
- Test: One-way ANOVA (two-tailed)
- H0: Mean `Avg_Rating` is equal across all cuisine types.
- H1: Mean `Avg_Rating` differs across at least one cuisine type.

In [18]:
cuisine_rating = restaurants.groupby('Cuisine_Type')['Avg_Rating'].mean().sort_values(ascending=False)
print(cuisine_rating.round(3))

groups16 = [g['Avg_Rating'].dropna().values for _, g in restaurants.groupby('Cuisine_Type')]
f16, p16 = f_oneway(*groups16)
print(f'\nOne-way ANOVA -> F = {f16:.3f}, p = {p16:.4g}')

Cuisine_Type
Italian     4.6390
Chinese     4.6260
Fast Food   4.6160
Indian      4.6130
Name: Avg_Rating, dtype: float64

One-way ANOVA -> F = 0.508, p = 0.6768


**Output Interpretation:** F = 0.508, p = 0.6768 — above α = 0.05.

**Validation Summary**
**Decision: Fail to reject H0.** Average rating does not differ significantly by cuisine type — the
top-ranked cuisine above is only marginally ahead and shouldn't be treated as a systematic quality
difference.

## Q17 — Low_Capacity_Flag vs Avg_Rating
**Hypothesis Formulation**
- Test: Independent-samples t-test (one-tailed, left)
- H0: Mean `Avg_Rating` is equal for Low_Capacity_Flag = Yes vs No.
- H1: Mean `Avg_Rating` is lower for Low_Capacity_Flag = Yes restaurants.

In [19]:
low    = restaurants.loc[restaurants['Low_Capacity_Flag'] == 'Yes', 'Avg_Rating']
notlow = restaurants.loc[restaurants['Low_Capacity_Flag'] == 'No',  'Avg_Rating']
print('Mean rating, Low_Capacity=Yes:', round(low.mean(), 3), '| No:', round(notlow.mean(), 3))

t17, p17 = ttest_ind(low, notlow, equal_var=False)
p17_one = p17 / 2 if t17 < 0 else 1 - p17 / 2
print(f'\nt = {t17:.3f}, one-tailed p = {p17_one:.4g}')

Mean rating, Low_Capacity=Yes: 4.584 | No: 4.649

t = -3.028, one-tailed p = 0.001381


**Output Interpretation:** t = -3.03, one-tailed p = 0.0014 — well below α = 0.05.

**Validation Summary**
**Decision: Reject H0.** Low-capacity-flagged restaurants rate significantly lower than others — capacity
constraints appear to hurt the customer experience, supporting a capacity-expansion or order-throttling
intervention for these restaurants.

## Q18 — Order_Capacity_Per_Day vs Actual Order Volume
**Hypothesis Formulation**
- Test: Pearson correlation (one-tailed, right)
- H0: No significant correlation exists (ρ = 0).
- H1: A significant positive correlation exists (ρ > 0).

In [20]:
rest_orders    = master.groupby('Restaurant_ID').size().rename('Actual_Orders')
rest_analysis  = restaurants.join(rest_orders, on='Restaurant_ID').dropna(subset=['Actual_Orders'])

r18, p18 = pearsonr(rest_analysis['Order_Capacity_Per_Day'], rest_analysis['Actual_Orders'])
p18_one = p18 / 2 if r18 > 0 else 1 - p18 / 2
print(f'Pearson r = {r18:.4f}, one-tailed p = {p18_one:.4g}')

print('\nAvg capacity/day by city/region:')
print(restaurants.groupby('City')['Order_Capacity_Per_Day'].mean().round(1))

Pearson r = -0.2625, one-tailed p = 1

Avg capacity/day by city/region:
City
Metropolitan   47.4000
Semi-Urban      1.3000
Urban          19.1000
Name: Order_Capacity_Per_Day, dtype: float64


**Output Interpretation:** r = -0.263 (weak **negative**, opposite of the hypothesized direction), one-tailed
p ≈ 1.0 — far above α = 0.05.

**Validation Summary**
**Decision: Fail to reject H0.** Stated capacity per day does **not** positively predict actual order
volume — if anything the relationship trends slightly negative — so listed `Order_Capacity_Per_Day` is not
a reliable proxy for real demand and shouldn't be used alone to plan capacity/expansion.

---
# Objective 5 — Order Economics, Discounts & Profitability Risk
Quantify how discounting, payment mode, festival demand spikes, and order cancellations/delays affect net revenue and profitability.

## Q19 — Discount_Applied vs Net_Revenue
**Hypothesis Formulation**
- Test: Pearson correlation (one-tailed, left)
- H0: No significant relationship exists between discount and net revenue.
- H1: Higher discount is associated with lower net revenue per order (ρ < 0).

In [21]:
r19, p19 = pearsonr(master['Discount_Applied'], master['Net_Revenue'])
p19_one = p19 / 2 if r19 < 0 else 1 - p19 / 2
print(f'Pearson r = {r19:.4f}, one-tailed p = {p19_one:.4g}')

master['Discount_Band'] = pd.cut(master['Discount_Applied'], bins=[-1, 0, 50, 100, 1e9],
                                  labels=['No Discount', 'Low (<=50)', 'Medium (51-100)', 'High (>100)'])
print('\nAvg Net Revenue by discount band:')
print(master.groupby('Discount_Band')['Net_Revenue'].mean().round(2))

Pearson r = -0.1744, one-tailed p = 0

Avg Net Revenue by discount band:
Discount_Band
No Discount       540.7900
Low (<=50)        508.3200
Medium (51-100)   459.7500
High (>100)       399.6900
Name: Net_Revenue, dtype: float64


**Output Interpretation:** r = -0.174 (weak-to-moderate negative), p is effectively 0 (< 1e-300, n = 100,000) —
far below α = 0.05.

**Validation Summary**
**Decision: Reject H0.** Higher discounts are significantly associated with lower net revenue per order —
confirmed further by the declining average revenue across discount bands above — so discount depth needs
a profitability cap in pricing policy.

## Q20 — Order_Status vs Region
**Hypothesis Formulation**
- Test: Chi-square test of independence (two-tailed)
- H0: `Order_Status` (Cancelled/Returned vs Delivered) is independent of `Region`.
- H1: `Order_Status` is associated with `Region`.

In [22]:
ct20 = pd.crosstab(master['Region'], master['Order_Status'])
status_pct = (ct20.div(ct20.sum(axis=1), axis=0) * 100).round(2)
print(status_pct)

chi20, p20, dof20, exp20 = chi2_contingency(ct20)
print(f'\nChi-square = {chi20:.2f}, dof = {dof20}, p = {p20:.4g}')

Order_Status  Cancelled  Delivered  Returned
Region                                      
Metropolitan     5.0000    92.9700    2.0300
Semi-Urban       4.8900    93.1000    2.0100
Urban            4.9600    93.0100    2.0300

Chi-square = 0.54, dof = 4, p = 0.9696


**Output Interpretation:** χ² = 0.54, dof = 4, p = 0.9696 — well above α = 0.05.

**Validation Summary**
**Decision: Fail to reject H0.** Cancellation/return rates are essentially uniform across regions — order
status risk is not a regional issue and should be addressed with a company-wide fix rather than a
region-targeted one.

---
# ✅ Overall Validation Summary — All 20 Hypotheses

| Q# | Objective | Test | Result | Decision |
|----|-----------|------|--------|----------|
| Q1/Q2 | 1 | One-way ANOVA | F=1.053, p=0.349 | Fail to reject H0 |
| Q3 | 1 | Trend regression (per region) | p≈0.067–0.079 all regions | Fail to reject H0 |
| Q4 | 1 | Independent t-test (per region) | p≈1e-22 to 1e-24 all regions | **Reject H0** |
| Q5 | 2 | One-way ANOVA | F=199.80, p=1.02e-86 | **Reject H0** |
| Q6 | 2 | Chi-square | χ²=16.23, p=0.093 | Fail to reject H0 |
| Q7 | 2 | Pearson r | r=0.048, p=4.5e-13 | **Reject H0** (weak effect) |
| Q8 | 2 | One-way ANOVA | F=1.107, p=0.331 | Fail to reject H0 |
| Q9 | 2 | Chi-square | χ²=2.88, p=0.824 | Fail to reject H0 |
| Q10 | 2 | Chi-square | χ²=46.71, p=4.1e-05 | **Reject H0** |
| Q11 | 3 | One-way ANOVA | F=1.058, p=0.366 | Fail to reject H0 |
| Q12 | 3 | Pearson r | r=0.376, p=1.2e-38 | **Reject H0** |
| Q13 | 3 | Chi-square | χ²=608.21, p=2.6e-130 | **Reject H0** (Semi-Urban = 100% delay rate — verify data quality) |
| Q14 | 3 | Pearson r | r=-0.002, p=0.244 | Fail to reject H0 |
| Q15 | 4 | Pearson r | r=-0.306, p=1.1e-13 | **Reject H0** |
| Q16 | 4 | One-way ANOVA | F=0.508, p=0.677 | Fail to reject H0 |
| Q17 | 4 | Independent t-test | t=-3.03, p=0.0014 | **Reject H0** |
| Q18 | 4 | Pearson r | r=-0.263, p≈1.0 | Fail to reject H0 |
| Q19 | 5 | Pearson r | r=-0.174, p≈0 | **Reject H0** |
| Q20 | 5 | Chi-square | χ²=0.54, p=0.970 | Fail to reject H0 |

**Coverage note:** all 20 hypotheses specified in the Phase 3 document were already present in the Phase 3
EDA notebook with correctly matched test types and directional H1s — verified question-by-question above —
so no extra hypotheses were needed beyond the original 20; this notebook adds the executed statistics and
the explicit reject/retain Validation Summary for each.

### Key business takeaways
- **Expand on volume, not price:** order value doesn't differ by region (Q1/Q2), so prioritize regions by
  raw order/revenue scale and festival responsiveness (Q4), not an assumed higher-spend region.
- **Loyalty spend isn't linear:** Loyal customers spend the most, but Returning < New (Q5) — Returning
  customers are a retention gap.
- **Efficiency and prep time both move ratings:** partner efficiency score (Q12) and restaurant prep time
  (Q15) are both significant rating drivers; delivery *time* alone (Q14) and vehicle type (Q11) are not.
- **Delay risk is regional, cancellation risk is not:** delay rate is strongly associated with region (Q13),
  but order cancellation/return status is not (Q20) — logistics investment should be regional, but
  cancellation fixes should be company-wide.
- **Capacity metric is unreliable:** `Order_Capacity_Per_Day` does not positively predict actual order
  volume (Q18) and low-capacity-flagged restaurants rate lower (Q17) — worth auditing how capacity is
  recorded.
- **Discounting has a real cost:** higher discounts significantly reduce net revenue per order (Q19) —
  pricing policy needs a discount ceiling tied to margin targets.
